# Social Media Meta-Analytics Data Cleaning Pipeline

**Objective:** To unify, clean, and standardize fragmented social media performance datasets (Meta, X/Twitter, LinkedIn, TikTok) into a single, analysis-ready source of truth.

**Portfolio Focus:** Python, Pandas, Data Integration, Data Cleaning Pipelines.

In [23]:
import pandas as pd
import numpy as np
import os

# Define local filenames for the fragmented datasets
meta_file = '/Users/skyy/Documents/Visual Studio Code/Repositories/data-cleaning/05_Social Media Analytics Cleanup/raw data/08a_meta_analytics.csv'
twitter_file = '/Users/skyy/Documents/Visual Studio Code/Repositories/data-cleaning/05_Social Media Analytics Cleanup/raw data/08b_twitter_analytics.csv'
linkedin_file = '/Users/skyy/Documents/Visual Studio Code/Repositories/data-cleaning/05_Social Media Analytics Cleanup/raw data/08c_linkedin_analytics.csv'
tiktok_file = '/Users/skyy/Documents/Visual Studio Code/Repositories/data-cleaning/05_Social Media Analytics Cleanup/raw data/08d_tiktok_analytics.csv'

output_filename = "cleaned_omnichannel_analytics.csv"

### Raw Data Analysis & Identified Issues

* **Fragmented Schemas:** Each platform uses proprietary naming conventions for identical metrics (e.g., Twitter's `favourites` vs. Meta's `likes`, TikTok's `video_views` vs. LinkedIn's `impressions`).
* **Timezone Inconsistencies:** The `post_date` column contains mixed timezone strings (`UTC`, `IST`), making time-series analysis impossible without normalization.
* **Polluted Data (Test Accounts):** The datasets contain records from `TEST001` and `INTERNAL02` accounts, which will artificially inflate engagement metrics and ruin analytics.
* **Missing Value Injection:** Concatenating datasets with differing architectures (like LinkedIn's `unique_views` or TikTok's lack of `shares`) will introduce `NaN` values that must be handled numerically.

In [24]:
# Ingestion: Load all datasets
df_meta = pd.read_csv(meta_file, skipinitialspace=True)
df_twit = pd.read_csv(twitter_file, skipinitialspace=True)
df_link = pd.read_csv(linkedin_file, skipinitialspace=True)
df_tik = pd.read_csv(tiktok_file, skipinitialspace=True)

print(f"Meta shape: {df_meta.shape}")
print(f"Twitter shape: {df_twit.shape}")
print(f"LinkedIn shape: {df_link.shape}")
print(f"TikTok shape: {df_tik.shape}")

Meta shape: (120, 9)
Twitter shape: (100, 8)
LinkedIn shape: (80, 8)
TikTok shape: (90, 7)


In [25]:
# 1. Schema Standardization (Column Mapping)
# Rename platform-specific metrics to a unified funnel nomenclature
df_twit = df_twit.rename(columns={
    'views': 'impressions', 
    'favourites': 'likes', 
    'replies': 'comments', 
    'retweets': 'shares'
})

df_link = df_link.rename(columns={
    'reactions': 'likes',
    'unique_views': 'reach' # Proxied for standard funnel
})

df_tik = df_tik.rename(columns={
    'video_views': 'impressions'
})

# 2. Data Integration (Concatenation)
df = pd.concat([df_meta, df_twit, df_link, df_tik], ignore_index=True)
print(f"Combined Shape: {df.shape}")

# 3. Temporal Normalization (FIXED)
# Map ambiguous timezone abbreviations to explicit numerical offsets
df['post_date'] = df['post_date'].str.replace(' IST', '+05:30', regex=False)
df['post_date'] = df['post_date'].str.replace(' UTC', '+00:00', regex=False)

# Now pandas can safely convert everything to a standardized UTC datetime object
df['post_date'] = pd.to_datetime(df['post_date'], format='mixed', utc=True)

# 4. Data Quality Filtering (Removing Internal/Test Accounts)
invalid_accounts = ['TEST001', 'INTERNAL02']
df = df[~df['account_id'].isin(invalid_accounts)]

# 5. Handling Concatenation NaNs
# Metrics that don't exist on certain platforms (e.g., shares on TikTok) become NaNs. Fill with 0.
engagement_metrics = ['impressions', 'reach', 'likes', 'comments', 'shares']
for col in engagement_metrics:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# Sort by date for chronological integrity
df = df.sort_values('post_date').reset_index(drop=True)

print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"Final dataset shape: {df.shape}")
df.head()

Combined Shape: (390, 9)
Remaining missing values: 0
Final dataset shape: (292, 9)


,post_id,account_id,post_date,impressions,reach,likes,comments,shares,platform
0,ME00066,ACC001,2024-01-01 23:00:00+00:00,16244,2233,1900,277,304,Meta
1,TT00023,ACC002,2024-01-02 12:30:00+00:00,24982,0,9852,142,0,TikTok
2,TT00033,ACC004,2024-01-02 18:30:00+00:00,55480,0,3840,359,0,TikTok
3,ME00031,ACC004,2024-01-02 20:00:00+00:00,19081,13522,81,60,241,Meta
4,LI00054,ACC002,2024-01-03 14:00:00+00:00,5708,5510,368,11,0,LinkedIn


In [26]:
# Export the finalized, clean omnichannel dataframe
df.to_csv(output_filename, index=False)
print(f"Data integration and cleaning complete. Saved securely at: {output_filename}")

Data integration and cleaning complete. Saved securely at: cleaned_omnichannel_analytics.csv
